# Finding model limitiations
This notebook is meant to generate plots of cases where inference goes badly, mostly due to inputs not matching the training data and/or model assumptions

In [ ]:
import sys

sys.path.append('..')

import matplotlib.pyplot as plt
import numpy as np
import torch

from src.flow_matching.helpers import choose_device
from src.flow_matching.loader import (
    load_model_info,
    load_trained_model,
    prob_path_from_config,
)
from src.flow_matching.neural_posterior import NeuralPosterior
from src.flow_matching.simulator import Model as BatchedModel

device = choose_device()

In [ ]:
# fill in desired job_id or directory name 
job_id = "15101905"

vector_field, classifier, config = load_trained_model(job_id, device)

transdimensional = not config['training']['fixed_N']
path = prob_path_from_config(config)
model_info = load_model_info(config, path=path, device=device)

In [ ]:
inf_params  = model_info['inf_params']
N           = model_info['N']
vector_dim  = model_info['vector_dim']
burstparams = model_info['burstparams']
mean        = model_info['mean']
std         = model_info['std']
YBKG        = config['path']['p_data']['init_params']['model_params']['ybkg']

In [ ]:
def downsample(profile, factor=1):
    """
    Reduce time resolution by merging bins.
    """
    for i in range(factor):
        bins_1 = profile[::2]
        bins_2 = profile[1::2]
        if len(bins_1) != len(bins_2):
            bins_1_new = torch.zeros_like(bins_2)
            bins_1_new = bins_1[:-1]
            bins_1 = bins_1_new
        profile = bins_1 + bins_2
    return profile 

In [ ]:
K = 500
t_max = 1.0
TIME = torch.linspace(0, t_max, K)
 
# standard burst parameter values when not sampled from prior
AMP  = 15
RISE = 0.03
SKEW = 3
N = 2

BURSTPARAMS = {
    't0'   : torch.linspace(0.2, t_max - 0.3, N).unsqueeze(0),
    'amp'  : torch.Tensor([np.log10(AMP)]).repeat(N).unsqueeze(0),
    'rise' : torch.Tensor([np.log10(RISE)]).repeat(N).unsqueeze(0),
    'skew' : torch.Tensor([SKEW]).repeat(N).unsqueeze(0)
}

MODELPARAMS = {
    'time' : TIME,
    'ncomp': N,
    'burstparams': BURSTPARAMS,
    'ybkg': YBKG
}

ncomp = torch.tensor([N], device=device).unsqueeze(0)
simulated_profile = BatchedModel(
            time=MODELPARAMS['time'], 
            ncomp=ncomp, 
            ybkg=MODELPARAMS['ybkg'], 
            burstparams=BURSTPARAMS,
            device=device
            ).get_flux()[0]

noisy_profile = simulated_profile + torch.randn_like(simulated_profile)

In [ ]:
factor = 1 
updated_bins = K // (2 ** factor)
print(updated_bins)
plt.plot(np.linspace(0, t_max, updated_bins), downsample(noisy_profile.cpu(), factor=factor), label='low res')
plt.plot(TIME, noisy_profile.cpu(), label='high res')

plt.legend()
plt.title('wrong resolution')

In [ ]:
learned_posterior = NeuralPosterior(
    vector_field,
    classifier,
    model_info,
    path,
    device=device
)

In [ ]:
num_samples = 1000
Ns, theta_samples = learned_posterior.sample(num_samples, noisy_profile, time_steps=200)
p_N = learned_posterior.p_N

In [ ]:
observed_counts = noisy_profile

In [ ]:
# posterior samples that include all sampled component numbers
posterior_param_samples = path.p_data.prior.samples_as_dict(theta_samples)

posterior_curve_samples = BatchedModel(
    device=device, 
    time=torch.linspace(0, t_max, observed_counts.shape[0]), 
    burstparams=posterior_param_samples, 
    ybkg=YBKG, 
    ncomp=Ns
        ).get_flux()

import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(11, 4))
gs = gridspec.GridSpec(1, 2, width_ratios=[1.5, 1])  # first subplot wider

ax1 = plt.subplot(gs[0])
ax2 = plt.subplot(gs[1])

ax1.plot(np.linspace(0, 1, observed_counts.shape[0]), observed_counts.flatten().cpu(),'k-', alpha=0.6,  label='observed flux')
sample_colors=['pink', 'yellow', 'orange', 'purple', 'green', 'cyan', 'red', 'grey', 'magenta', 'blue']
# for i in range(num_samples):
#     random_idx = i#torch.randint(0, num_samples, size=(1,)).item()
#     ax1.plot(
#         np.linspace(0, 1, observed_counts.shape[0]), 
#         posterior_curve_samples[random_idx].cpu(), 
#         alpha=0.07, 
#         color='red',
#         # alpha=1,
#         # color = sample_colors[int(Ns[random_idx].item()) - 1], 
#         label='posterior samples' if i==0 else ''
#         )


# plot mean curve
plot_mean =True 
if plot_mean:
    mean_profile = torch.mean(posterior_curve_samples[0:num_samples], dim=0).cpu()
    time_ = np.linspace(0, t_max, observed_counts.shape[0])
    ax1.plot(time_, mean_profile, color='white', label='mean profile', linewidth=1)

    # 95% percentile
    q1 = torch.quantile(posterior_curve_samples, 0.025, dim=0).cpu()
    q3 = torch.quantile(posterior_curve_samples, 0.975, dim=0).cpu()
    iqr = q3 - q1
    ax1.fill_between(time_, q1, q3, color='red', alpha=0.5, label='std profile')

# name = filepath.split('/')[-1].split('_')[0]
# ax1.set_title(name, fontsize=16)
ax1.grid(linestyle='dotted')
ax1.set_ylabel('SNR', fontsize=14)
ax1.set_xlabel('t', fontsize=14)
ax1.set_xlim(0,1)
ax1.tick_params(axis='both', labelsize=14)

ax2.bar(
    range(1, len(p_N.cpu().detach().numpy())+1), 
    height=p_N.cpu().detach().numpy(), 
    # color=sample_colors,
    color='lightsteelblue', 
    edgecolor='black',    # color of the bar edge
    linewidth=1,
    label=range(1, 21)
    )
# ax2.legend()
ax2.set_xlabel("$N$", fontsize=14)
ax2.set_ylabel('$p(N\mid y)$', fontsize=14, labelpad=15)
ax2.tick_params(axis='both', labelsize=14)
plt.xlim(0, 20.5)
import matplotlib.ticker as mticker
ax2.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.savefig(f'poor_inference_at_low_res.png', dpi=300)

plt.show()
print(observed_counts.shape)